In [3]:
import findspark, os
import sys

# Change to path on your local machine.
os.environ["SPARK_HOME"] = "/Users/andy/Documents/spark-4.0.1-bin-hadoop3"
findspark.init()

from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.functions import broadcast

py = sys.executable  # the Python of this notebook (e.g., .../envs/yourenv/bin/python)
os.environ["PYSPARK_DRIVER_PYTHON"] = py
os.environ["PYSPARK_PYTHON"] = py

spark = SparkSession.getActiveSession() or (
    SparkSession.builder
    .appName("A3")
    .master("local[*]")
    .config("spark.driver.memory", "8g")           
    .config("spark.sql.shuffle.partitions","400")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.pyspark.driver.python", py)
    .config("spark.pyspark.python", py)
    .config("spark.executorEnv.PYSPARK_PYTHON", py)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

sc = spark.sparkContext
print("Spark:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/14 13:39:08 WARN Utils: Your hostname, MacBook-Air-uzivatela-Andy.local, resolves to a loopback address: 127.0.0.1; using 10.10.90.242 instead (on interface en0)
25/11/14 13:39:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/14 13:39:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 4.0.1


In [6]:
data_path = "/Users/andy/Documents/cs651/a5"

In [7]:
!wc {data_path}/corpus-enwiki-sample.txt

 10805214 330541791 1954717986 /Users/andy/Documents/cs651/a5/corpus-enwiki-sample.txt


In [8]:
import argparse
import re
from collections import Counter
from pyspark import SparkConf, SparkContext

# Tokenizer function
def tokenize(line):
    TOKEN_RE = re.compile(r'(^[^a-z]+|[^a-z]+$)')
    out = []
    for tok in line.split():
        w = TOKEN_RE.sub('', tok.lower())
        if w:
            out.append(w)
    return out

# Parse input lines of format: "docid<TAB>text"
def parse_line(line):
    parts = line.split("\t", 1)
    if len(parts) == 2:
        return int(parts[0]), parts[1]
    else:
        return None


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('-input', required=True)
    ap.add_argument('-output', required=True)
    args = ap.parse_args()

    conf = SparkConf().setAppName('build_index_uncompressed')
    sc = SparkContext(conf=conf)

    # Read input file: each line is "docid\ttext"
    rdd = sc.textFile(args.input)

    # BEGIN OF YOUR CODE
    # Parse lines into (docid, text)
    docs = rdd.map(parse_line).filter(lambda x: x is not None)

    # Mapper phase: emit (term, (docid, tf))
    # For each document, tokenize and count term frequencies
    term_doc_pairs = docs.flatMap(lambda x: [((term, x[0]), 1) for term in tokenize(x[1])])
    term_doc_tf = term_doc_pairs.reduceByKey(lambda a, b: a + b)  # sum term frequencies
    term_to_posting = term_doc_tf.map(lambda x: (x[0][0], (x[0][1], x[1])))

    # Reducer phase: group postings by term, sort postings by docid
    postings_lists = (
        term_to_posting
        .groupByKey()
        .mapValues(lambda postings: sorted(list(postings), key=lambda p: p[0]))
    )

    # END OF YOUR CODE

    postings_lists.saveAsTextFile(args.output)
    sc.stop()


if __name__ == '__main__':
    main()


usage: ipykernel_launcher.py [-h] -input INPUT -output OUTPUT
ipykernel_launcher.py: error: the following arguments are required: -input, -output


SystemExit: 2

/opt/anaconda3/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
